**Wir befinden uns hier in einem Python 3.8, aber 3DINO ist mit Python 3.9 gebastelt, also müssen wir testen, ob es überhaupt funktioniert.**  
Anpassungen:
* Wörkaround für `args.arch.removesuffix` in `dinov2/models/__init__.py`

In [1]:
# Changes introduced to DINO to make it work with our dataset and on Python3.8
! cd DINO; git diff

diff --git a/dinov2/data/loaders.py b/dinov2/data/loaders.py
index b476bb9..d3c0de3 100644
--- a/dinov2/data/loaders.py
+++ b/dinov2/data/loaders.py
@@ -113,6 +113,10 @@ def make_segmentation_dataset_3d(
         datalist_path = os.path.join(base_directory, 'BraTS_100_datalist.json')
         class_num = 3
         input_channels = 4
+    elif dataset_name == 'hip_ultrasound':
+        datalist_path = os.path.join(base_directory, 'hip_ultrasound_100_datalist.json')
+        class_num = 5
+        input_channels = 1
     elif dataset_name == 'LA-SEG':
         datalist_path = os.path.join(base_directory, 'LA-SEG_100_datalist.json')
         class_num = 2
diff --git a/dinov2/eval/segmentation3d.py b/dinov2/eval/segmentation3d.py
index f6e4675..91e05fc 100644
--- a/dinov2/eval/segmentation3d.py
+++ b/dinov2/eval/segmentation3d.py
@@ -115,8 +115,14 @@ def train_iter(model, batch, optimizer, scheduler, loss_function, scaler):
 def val_iter(model, batch, metric, image_size, batch_size, overl

In [2]:
import sys
from pathlib import Path
import subprocess
import pandas as pd
import json
import os
from datetime import datetime

In [3]:
# Projekt Konstanten

PROJECT_DIR = Path("/data_rechenknecht03_2/students/boehmerle")
OUTPUT_DIR = PROJECT_DIR / "outputs"
REPO_DIR = PROJECT_DIR / "DINO" # Python Packages können nicht mit einer Zahl anfangen, also nutzen wir nur DINO, auch wenn es eigentlich 3DINO ist.

DATASET_NAME = "hip_ultrasound"
DATALIST_DIR = PROJECT_DIR / "datalists"

CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "3dino_vit_weights.pth"

MAX_WORKERS = min(8, os.cpu_count() or 1)

In [4]:
# 3DINO klonen und importieren

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/AICONSlab/3DINO.git",
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    print(f"Repository existiert bereits: {REPO_DIR}")

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print(f"3DINO-Commit: {commit}")

import DINO.dinov2

Repository existiert bereits: /data_rechenknecht03_2/students/boehmerle/DINO
3DINO-Commit: 85bd4435c1b2ada41cd34cd15cad17c4d3c88d89


In [5]:
# Datensatz für 3DINO vorbereiten
DATALIST_DIR.mkdir(exist_ok=True)

manifest_path = OUTPUT_DIR / "manifest_with_split.csv"
manifest = pd.read_csv(manifest_path)

datalist = {}

for split_name in ["training", "validation", "test"]:
    subset = (
        manifest
        .query("pair_status == 'ok' and split == @split_name")
        .sort_values(["case_id", "side", "sweep_id"])
    )

    datalist[split_name] = [
        {
            "image": row.image_path,
            "label": row.label_path,
        }
        for row in subset.itertuples(index=False)
    ]

datalist_path = DATALIST_DIR / f"{DATASET_NAME}_100_datalist.json"

with open(datalist_path, "w", encoding="utf-8") as file:
    json.dump(datalist, file, indent=2)

print(f"Gespeichert: {datalist_path}")
for split_name, entries in datalist.items():
    print(f"{split_name}: {len(entries)} Sweeps")

print("\nErster Trainingseintrag:")
print(json.dumps(datalist["training"][0], indent=2))

Gespeichert: /data_rechenknecht03_2/students/boehmerle/datalists/hip_ultrasound_100_datalist.json
training: 132 Sweeps
validation: 26 Sweeps
test: 32 Sweeps

Erster Trainingseintrag:
{
  "image": "/data_rechenknecht03_2/students/boehmerle/img/0EXN9_sweep04_links_1x_p_a_rec.nii.gz",
  "label": "/data_rechenknecht03_2/students/boehmerle/seg/0EXN9_sweep04_links_1x_p_a_seg.nii.gz"
}


In [6]:
def run_segmentation(
    output_dir,
    batch_size=1,
    num_workers=MAX_WORKERS,
    segmentation_head="Linear",
    epochs=1,
    epoch_length=1,
    learning_rate=1e-4):

    run_env = os.environ.copy()
    run_env["PYTHONPATH"] = str(REPO_DIR)
    run_env["CUDA_VISIBLE_DEVICES"] = "0"
    # run_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # Ist wohl hier leider nicht verfügbar.
    # Eigentlich keine gute Idee, aber in diesem Fall wird uns der Log vollgespammt.
    run_env["PYTHONWARNINGS"] = "ignore:You are using `torch.load` with `weights_only=False`:FutureWarning"

    full_output_dir = OUTPUT_DIR / output_dir
    full_output_dir.mkdir(parents=True, exist_ok=True)

    log_path = full_output_dir / "run.log"
    
    with open(log_path, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            [
                sys.executable,
                "dinov2/eval/segmentation3d.py",
                "--config-file", "dinov2/configs/train/vit3d_highres.yaml",
                "--dataset-name", "hip_ultrasound",
                "--dataset-percent", "100",
                "--base-data-dir", str(DATALIST_DIR),
                "--pretrained-weights", str(CHECKPOINT_PATH),
                "--segmentation-head", "Linear",
                "--image-size", "96",
                "--resize-scale", "1.0",
                "--eval-iters", str(max(1, int(epoch_length / 2))),
                "--warmup-iters", "0",
                "--segmentation-head", segmentation_head,
                "--epochs", str(epochs),
                "--epoch-length", str(epoch_length),
                "--batch-size", str(batch_size),
                "--num-workers", str(num_workers),
                "--learning-rate", str(learning_rate),
                "--output-dir", str(full_output_dir),
                "--cache-dir", str(PROJECT_DIR / "cache" / "hip_ultrasound"),
            ],
            cwd=REPO_DIR,
            text=True,
            env=run_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1
        )

        for line in iter(process.stdout.readline, ""):
            print(f"[{datetime.now().time()}]", line, end="")
            log_file.write(line)
            log_file.flush()

        process.stdout.close()

        return [process.wait(), log_file]

In [ ]:
# Tut das ganze überhaut?
[return_code, log_path] = run_segmentation("hip_ultrasound_smoke_linear", batch_size=1, epochs=1, epoch_length=1, learning_rate=1e-4)

print(f"Finished at {datetime.now().time()}")
print(f"Return-Code: {return_code}")
print(f"Logdatei: {log_path}")

[12:30:24.779826] I20260924 12:30:24 3266 dinov2 config.py:60] git:
[12:30:24.781063]   sha: 85bd4435c1b2ada41cd34cd15cad17c4d3c88d89, status: has uncommitted changes, branch: main
[12:30:24.781130] 
[12:30:24.781213] I20260924 12:30:24 3266 dinov2 config.py:61] base_data_dir: /data_rechenknecht03_2/students/boehmerle/datalists
[12:30:24.781257] batch_size: 1
[12:30:24.781296] cache_dir: /data_rechenknecht03_2/students/boehmerle/cache/hip_ultrasound
[12:30:24.781334] config_file: dinov2/configs/train/vit3d_highres.yaml
[12:30:24.781373] dataset_name: hip_ultrasound
[12:30:24.781412] dataset_percent: 100
[12:30:24.781453] epoch_length: 1
[12:30:24.781491] epochs: 1
[12:30:24.781530] eval_iters: 1
[12:30:24.781571] image_size: 96
[12:30:24.781610] learning_rate: 0.0001
[12:30:24.781648] num_workers: 8
[12:30:24.781687] opts: ['train.output_dir=/data_rechenknecht03_2/students/boehmerle/outputs/hip_ultrasound_smoke_linear', 'train.cache_dir=/data_rechenknecht03_2/students/boehmerle/cache/h

In [ ]:
# Ein größerer Test mit etwas mehr Daten und mehr Epochen:
[return_code, log_path] = run_segmentation("hip_ultrasound_smoke_linear", batch_size=5, epochs=100, epoch_length=10, learning_rate=1e-4)

print(f"Finished at {datetime.now().time()}")
print(f"Return-Code: {return_code}")
print(f"Logdatei: {log_path}")